In [1]:
# -*- coding: utf-8 -*-
"""
Incendis forestals a Catalunya (1998-2024) — Visualitzacions Finals Q1-Q4
Information Visualization | Projecte 1
"""

import copy
import json
import numpy as np
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

# %% ===================================================================
# 0. CÀRREGA I NETEJA DE DADES
# =======================================================================
DATA_PATH = "data/incendis_final.csv"
COMARQUES_MAP_PATH = 'data/divisions-administratives-v2r2-comarques-500000-20260120.json'

df = pd.read_csv(DATA_PATH)

num_cols = [
    'HAARBRADES', 'HANOARBRAD', 'SuperficieTotalForestal',
    'HANOFOREST', 'SuperficieAgricola',
    'OtrasSuperficiesNoforestales', 'SuperficieTotalNoForestal'
]
for c in num_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors='coerce')

df['SuperficieTotal'] = (
    df['SuperficieTotalForestal'].fillna(0)
    + df['SuperficieTotalNoForestal'].fillna(0)
)


def any_des_de_data(serie):
    dates = pd.to_datetime(serie, dayfirst=True, errors='coerce', format='mixed')
    return dates.dt.year


any_1 = any_des_de_data(df['Data incendi'])
any_2 = any_des_de_data(df['Data incendi'])
any_campania = df['Campania'].astype(str).str.extract(r'(\d{4})\s*$')[0].astype(float)
df['Any'] = any_1.combine_first(any_2).combine_first(any_campania)

df = df.dropna(subset=['Nom comarca', 'Any']).copy()
df['Any'] = df['Any'].astype(int)
df = df[(df['Any'] >= 1998) & (df['Any'] <= 2024)]

# Unifiquem causes buides o "Desconeguda" amb la categoria "Desconocida" real del CSV
_causes_valides = df['Causa'].dropna().astype(str).str.strip()
_candidata_desconeguda = next(
    (c for c in _causes_valides.unique() if 'esconoc' in c.lower()),
    next((c for c in _causes_valides.unique() if 'esconeg' in c.lower()), 'Desconocida')
)
df['Causa'] = df['Causa'].fillna(_candidata_desconeguda).astype(str).str.strip()
df['Causa'] = df['Causa'].replace({
    'nan': _candidata_desconeguda,
    '': _candidata_desconeguda,
    'Desconeguda': _candidata_desconeguda,
    'desconeguda': _candidata_desconeguda
})

for c in ['Nom comarca', 'Nom vegueria']:
    if c in df.columns:
        df[c] = df[c].astype(str).str.strip()

correccions_comarca = {
    'Aran': "Val d'Aran",
    'Montisà': 'Montsià',
    "Val d'Aran": "Val d'Aran"
}
df['Nom comarca'] = df['Nom comarca'].replace(correccions_comarca)

top_causes = df['Causa'].value_counts().nlargest(6).index
df['CausaAgrupada'] = df['Causa'].where(df['Causa'].isin(top_causes), 'Altres')
causa_domain = sorted(df['CausaAgrupada'].unique())
causa_scale = alt.Scale(domain=causa_domain, scheme='dark2')


# --- GeoJSON i càlcul d'aspect ratio real ---
geojson_path = (
    COMARQUES_MAP_PATH
)
with open(geojson_path, encoding='utf-8') as f:
    geo_dict = json.load(f)
comarques_geo = alt.Data(values=geo_dict['features'])
noms_geojson = {feat['properties']['NOMCOMAR'] for feat in geo_dict['features']}


def _calcular_bbox(geo_d):
    xs, ys = [], []

    def recorre(coords):
        if isinstance(coords[0], (int, float)):
            xs.append(coords[0])
            ys.append(coords[1])
        else:
            for c in coords:
                recorre(c)

    for feat in geo_d['features']:
        recorre(feat['geometry']['coordinates'])
    return min(xs), max(xs), min(ys), max(ys)


_minx, _maxx, _miny, _maxy = _calcular_bbox(geo_dict)
ASPECT_RATIO = (_maxx - _minx) / (_maxy - _miny)


def mida_mapa(amplada):
    return amplada, round(amplada / ASPECT_RATIO)


PROJECCIO = 'identity'
REFLECTEIX_Y = True


def titol_rt(titol, subtitol):
    return alt.TitleParams(
        text=titol,
        subtitle=subtitol,
        anchor='start',
        fontSize=20,
        fontWeight='bold',
        subtitleFontSize=13,
        subtitleColor='#555555',
        offset=12
    )

In [2]:
# %% ===================================================================
# Q2 · Triennis (q2v1b) + Quadradets en 3 files etiquetades (Petits/Mitjans=20 focs, Grans=1 foc)
# =======================================================================
df_q2 = df[df['Any'] <= 2022].copy()
bins_trienni = [1998, 2001, 2004, 2007, 2010, 2013, 2016, 2019, 2023]
labels_trienni = [
    '1998-2000', '2001-2003', '2004-2006', '2007-2009',
    '2010-2012', '2013-2015', '2016-2018', '2019-2022'
]
df_q2['Trienni'] = pd.cut(df_q2['Any'], bins=bins_trienni, labels=labels_trienni, right=False)

AMPLADA_BARRA_Q2 = 540
ALCADA_BARRA_Q2 = 66
AMPLADA_LABEL_Q2 = 90
AMPLADA_BOLA_TOTAL_Q2 = 85
AMPLADA_REF_TRAM_Q2 = 105
AMPLADA_PICTO_Q2 = 360

tram_q2_domain = ['Petits (<1 ha)', 'Mitjans (1-500 ha)', 'Grans (>=500 ha)']
tram_q2_colors = alt.Scale(
    domain=tram_q2_domain,
    range=['#fdae61', '#f46d43', '#a50026']
)


def quadradets_periode_q2(subdf, periode):
    files = []
    # Fila 0 (a dalt): Grans (>= 500 ha) -> 1 quadradet = 1 incendi real
    grans = subdf[subdf['SuperficieTotal'] >= 500].sort_values('SuperficieTotal', ascending=False)
    for i, (_, r) in enumerate(grans.iterrows()):
        mun = r.get('Nom municipi', r.get('Nom comarca', ''))
        if pd.isna(mun) or str(mun).strip() == '':
            mun = r.get('Nom comarca', '')
        files.append({
            'Periode': periode,
            'Tram': 'Grans (>=500 ha)',
            'y_row': 0,
            'col': i,
            'mida_sq': 115,
            'Representa': '1 Gran Incendi Forestal',
            'Detall': f"{mun} ({int(r['Any'])}): {r['SuperficieTotal']:,.1f} ha"
        })

    # Fila 1 (al mig): Mitjans (1-500 ha) -> 1 columna/quadradet = 20 incendis
    n_mitjans = int(((subdf['SuperficieTotal'] >= 1) & (subdf['SuperficieTotal'] < 500)).sum())
    n_sq_mitjans = int(np.ceil(n_mitjans / 20.0)) if n_mitjans > 0 else 0
    for i in range(n_sq_mitjans):
        files.append({
            'Periode': periode,
            'Tram': 'Mitjans (1-500 ha)',
            'y_row': 1,
            'col': i,
            'mida_sq': 80,
            'Representa': '20 incendis mitjans (1-500 ha)',
            'Detall': f"Total al trienni: {n_mitjans:,} incendis mitjans"
        })

    # Fila 2 (a baix): Petits (< 1 ha) -> 1 columna/quadradet = 20 incendis
    n_petits = int((subdf['SuperficieTotal'] < 1).sum())
    n_sq_petits = int(np.ceil(n_petits / 20.0)) if n_petits > 0 else 0
    for i in range(n_sq_petits):
        files.append({
            'Periode': periode,
            'Tram': 'Petits (<1 ha)',
            'y_row': 2,
            'col': i,
            'mida_sq': 55,
            'Representa': '20 conats (< 1 ha)',
            'Detall': f"Total al trienni: {n_petits:,} conats"
        })

    return pd.DataFrame(files)


df_sq_q2 = pd.concat(
    [quadradets_periode_q2(df_q2[df_q2['Trienni'] == p], p) for p in labels_trienni],
    ignore_index=True
)
max_col_q2 = max(int(df_sq_q2['col'].max()) + 2, 45)

# Càlcul manual d'apilament de les barres de causes per trienni
df_causa_q2 = (
    df_q2.groupby(['Trienni', 'CausaAgrupada'], as_index=False, observed=True)
    .size().rename(columns={'size': 'n'})
)
parts_q2 = []
for idx_p, p in enumerate(labels_trienni):
    sub = df_causa_q2[df_causa_q2['Trienni'] == p].set_index('CausaAgrupada').reindex(causa_domain).reset_index()
    sub['n'] = sub['n'].fillna(0)
    tot = sub['n'].sum()
    sub['pct'] = sub['n'] / tot if tot else 0
    sub['x1'] = sub['pct'].cumsum()
    sub['x0'] = sub['x1'] - sub['pct']
    sub['x_mig'] = (sub['x0'] + sub['x1']) / 2
    sub['Trienni'] = p
    sub['label_txt'] = sub['pct'].apply(
        lambda v: (f"{round(v * 100)}%" if idx_p == 0 else f"{round(v * 100)}") if v >= 0.045 else ""
    )
    parts_q2.append(sub)
df_causa_q2 = pd.concat(parts_q2, ignore_index=True)

# Capçaleres de columna Q2
cap_q2_causes = alt.Chart(pd.DataFrame({'CausaAgrupada': causa_domain})).mark_square(size=180, opacity=0).encode(
    color=alt.Color(
        'CausaAgrupada:N', scale=causa_scale, title='Causa:',
        legend=alt.Legend(
            orient='top', direction='horizontal', columns=3,
            titleOrient='left', titleFontSize=13, labelFontSize=12,
            symbolType='circle', symbolSize=180, symbolOpacity=1
        )
    )
).properties(width=AMPLADA_BARRA_Q2, height=1)

cap_q2_any = alt.Chart(pd.DataFrame({'t': ['TRIENNI']})).mark_text(
    fontSize=13, fontWeight='bold', color='#222222'
).encode(text='t:N').properties(width=AMPLADA_LABEL_Q2, height=1)

cap_q2_tot = alt.Chart(pd.DataFrame({'t': ["Total focs"]})).mark_text(
    fontSize=12, fontWeight='bold', color='#222222'
).encode(text='t:N').properties(width=AMPLADA_BOLA_TOTAL_Q2, height=1)

cap_q2_ref = alt.Chart(pd.DataFrame({'t': ["Categoria mida"]})).mark_text(
    fontSize=12, fontWeight='bold', align='right', color='#222222'
).encode(text='t:N').properties(width=AMPLADA_REF_TRAM_Q2, height=1)

cap_q2_sq = alt.Chart(pd.DataFrame({'Tram': tram_q2_domain})).mark_square(size=180, opacity=0).encode(
    color=alt.Color(
        'Tram:N', scale=tram_q2_colors, title='Quadradets (Petits/Mitjans: 1 col = 20 focs | Grans: 1 = 1 GIF):',
        legend=alt.Legend(
            orient='top', direction='horizontal',
            titleOrient='top', titleFontSize=12, labelFontSize=12,
            symbolType='square', symbolSize=160, symbolOpacity=1
        )
    )
).properties(width=AMPLADA_PICTO_Q2, height=1)

files_q2 = [alt.hconcat(cap_q2_causes, cap_q2_any, cap_q2_tot, cap_q2_ref, cap_q2_sq, spacing=6)]

df_etiquetes_files_q2 = pd.DataFrame({
    'y_row': [0, 1, 2],
    'label_fila': ['Grans (>=500 ha)', 'Mitjans (1-500 ha)', 'Petits (<1 ha)']
})

for p in labels_trienni:
    sub_b = df_causa_q2[df_causa_q2['Trienni'] == p]
    c_barra = (
        alt.Chart(sub_b).mark_bar(height=ALCADA_BARRA_Q2).encode(
            x=alt.X('x0:Q', scale=alt.Scale(domain=[0, 1], nice=False), axis=None),
            x2='x1:Q',
            color=alt.Color('CausaAgrupada:N', scale=causa_scale, legend=None),
            tooltip=[
                alt.Tooltip('CausaAgrupada:N', title='Causa'),
                alt.Tooltip('n:Q', title="Nombre d'incendis", format=','),
                alt.Tooltip('pct:Q', title='Percentatge', format='.1%')
            ]
        ) + alt.Chart(sub_b[sub_b['pct'] >= 0.045]).mark_text(
            color='white', fontSize=13, fontWeight='bold', baseline='middle'
        ).encode(
            x=alt.X('x_mig:Q', scale=alt.Scale(domain=[0, 1], nice=False)),
            text='label_txt:N'
        )
    ).properties(width=AMPLADA_BARRA_Q2, height=ALCADA_BARRA_Q2 + 6)

    c_any = alt.Chart(pd.DataFrame({'t': [p]})).mark_text(
        fontSize=17, fontWeight='bold', align='center', color='#111111'
    ).encode(text='t:N').properties(width=AMPLADA_LABEL_Q2, height=ALCADA_BARRA_Q2 + 6)

    tot_p = int((df_q2['Trienni'] == p).sum())
    df_tot_p = pd.DataFrame({'x': [0], 't': [tot_p]})
    c_tot = (
        alt.Chart(df_tot_p).mark_circle(size=2200, color='#4a4a4a', opacity=0.92).encode(
            x=alt.X('x:Q', scale=alt.Scale(domain=[-1, 1]), axis=None)
        ) + alt.Chart(df_tot_p).mark_text(color='white', fontSize=12, fontWeight='bold').encode(
            x=alt.X('x:Q', scale=alt.Scale(domain=[-1, 1]), axis=None),
            text=alt.Text('t:Q', format=',')
        )
    ).properties(width=AMPLADA_BOLA_TOTAL_Q2, height=ALCADA_BARRA_Q2 + 6)

    # Referència explícita de les 3 files ocupant tota la vertical de la barra
    c_ref_files = alt.Chart(df_etiquetes_files_q2).mark_text(
        align='right', fontSize=11, fontWeight='bold', color='#444444'
    ).encode(
        y=alt.Y('y_row:Q', scale=alt.Scale(domain=[-0.5, 2.5], nice=False), axis=None),
        text='label_fila:N'
    ).properties(width=AMPLADA_REF_TRAM_Q2, height=ALCADA_BARRA_Q2 + 6)

    # Quadradets amb separació mínima en 3 files que ocupen tota la vertical de la barra
    sub_sq = df_sq_q2[df_sq_q2['Periode'] == p]
    guies_files = alt.Chart(df_etiquetes_files_q2).mark_rule(
        color='#eeeeee', strokeDash=[2, 2]
    ).encode(
        y=alt.Y('y_row:Q', scale=alt.Scale(domain=[-0.5, 2.5], nice=False), axis=None)
    )
    punts_sq = alt.Chart(sub_sq).mark_square(opacity=0.95, stroke='white', strokeWidth=0.3).encode(
        x=alt.X('col:Q', scale=alt.Scale(domain=[-0.8, max_col_q2], nice=False), axis=None),
        y=alt.Y('y_row:Q', scale=alt.Scale(domain=[-0.5, 2.5], nice=False), axis=None),
        size=alt.Size('mida_sq:Q', scale=alt.Scale(type='identity'), legend=None),
        color=alt.Color('Tram:N', scale=tram_q2_colors, legend=None),
        tooltip=[
            alt.Tooltip('Tram:N', title='Fila / Mida'),
            alt.Tooltip('Representa:N', title='Equivalència'),
            alt.Tooltip('Detall:N', title='Detall')
        ]
    )
    c_sq = (guies_files + punts_sq).properties(width=AMPLADA_PICTO_Q2, height=ALCADA_BARRA_Q2 + 6)

    files_q2.append(
        alt.hconcat(c_barra, c_any, c_tot, c_ref_files, c_sq, spacing=6).resolve_scale(color='independent')
    )

q2_final = alt.vconcat(*files_q2, spacing=4).properties(
    title=titol_rt(
        "Q2 · Evolució per triennis de les causes d'incendi i desglossament en 3 files per severitat (1998-2022)",
        "A la dreta, cada fila mostra els incendis Grans (1 quadradet = 1 GIF), Mitjans (1 quadradet = 20 focs) i Petits (1 quadradet = 20 conats)"
    )
).configure_view(strokeWidth=0).configure_axis(grid=False)
q2_final.save('outputs/Q2.json')